# 04 — Anomali (Uç Değer) Temizleme

Bu adımda winsorize/clip sınırları ile uç değerler kontrol altına alınır ve alan bazlı özel kurallar uygulanır (ör. `dti`, `revol_util`).  
Kullanılan sınırlar tekrar üretilebilirlik için artifact olarak saklanır.


In [8]:
import sys
sys.path.append('../../')

import json
import pandas as pd
from src.io.save_artifacts import load_parquet, save_parquet
from src.cleaning.outlier_rules import (
    apply_outlier_rules,
    compute_clip_bounds,
    OZEL_KURALLAR
)

df = load_parquet('../../data/processed/lending_club_no_missing.parquet')
print(f"Başlangıç shape: {df.shape}")

📂 Parquet okunuyor: ../../data/processed/lending_club_no_missing.parquet
✅ Okundu!
   Satır : 1,345,350
   Sütun : 77
   Süre  : 0.2 saniye
Başlangıç shape: (1345350, 77)


In [9]:
# Winsorize adaylarını yükle
with open('../../artifacts/winsorize_candidates.json') as f:
    winsor_kolonlar = json.load(f)

winsor_mevcut = [c for c in winsor_kolonlar if c in df.columns]
print(f"Winsorize uygulanacak: {len(winsor_mevcut)} kolon")
print(f"Listede olup veri setinde yok: "
      f"{len(winsor_kolonlar) - len(winsor_mevcut)}")

Winsorize uygulanacak: 19 kolon
Listede olup veri setinde yok: 13


In [10]:
# ÖNEMLİ NOT:
# Şu an tüm veri üzerinde clip bounds hesaplıyoruz.
# Modelleme adımında compute_clip_bounds(df_train) kullanılacak.

clip_bounds = compute_clip_bounds(
    df,
    kolonlar=winsor_mevcut,
    alt_percentile=0.01,
    ust_percentile=0.99
)

print("── Clip Bounds (ilk 10) ─────────────────────")
for k, v in list(clip_bounds.items())[:10]:
    print(f"   {k:<35}: [{v['alt']:.2f}, {v['ust']:.2f}]")

── Clip Bounds (ilk 10) ─────────────────────
   delinq_2yrs                        : [0.00, 4.00]
   inq_last_6mths                     : [0.00, 4.00]
   pub_rec                            : [0.00, 2.00]
   revol_bal                          : [171.00, 94553.51]
   tot_coll_amt                       : [0.00, 4698.51]
   total_rev_hi_lim                   : [2900.00, 154200.00]
   avg_cur_bal                        : [476.00, 72425.02]
   bc_open_to_buy                     : [0.00, 73144.51]
   mo_sin_rcnt_rev_tl_op              : [0.00, 83.00]
   mo_sin_rcnt_tl                     : [0.00, 42.00]


In [11]:
# Clip uygula
df = apply_outlier_rules(
    df,
    kolonlar=winsor_mevcut,
    clip_bounds=clip_bounds
)

── Uç Değer Düzeltme (clip) ─────────────────
   ✅ 19 kolona uygulandı
   Alt: %1  Üst: %99


In [12]:
# Özel anomali kurallarını göster ve uygula
print(" Özel Anomali Kuralları ")
for kolon, kural in OZEL_KURALLAR.items():
    if kolon not in df.columns:
        continue
    onceki_min = df[kolon].min()
    onceki_max = df[kolon].max()

    if kural.get('min') is not None:
        df.loc[df[kolon] < kural['min'], kolon] = kural['min']
    if kural.get('max') is not None:
        df.loc[df[kolon] > kural['max'], kolon] = kural['max']

    print(f"   {kolon:<20} "
          f"min: {onceki_min:.1f}→{df[kolon].min():.1f}  "
          f"max: {onceki_max:.1f}→{df[kolon].max():.1f}")

 Özel Anomali Kuralları 
   annual_inc           min: 0.0→0.0  max: 10999200.0→10999200.0
   dti                  min: -1.0→0.0  max: 999.0→100.0
   revol_util           min: 0.0→0.0  max: 892.3→100.0
   loan_amnt            min: 500.0→500.0  max: 40000.0→40000.0


In [13]:
# clip_bounds artifact olarak kaydet
clip_bounds_json = {
    k: {'alt': float(v['alt']), 'ust': float(v['ust'])}
    for k, v in clip_bounds.items()
}

with open('../../artifacts/clip_bounds.json', 'w') as f:
    json.dump(clip_bounds_json, f, indent=2)

print("✅ artifacts/clip_bounds.json kaydedildi")

✅ artifacts/clip_bounds.json kaydedildi


In [14]:
save_parquet(df, '../../data/processed/lending_club_clean.parquet')

print("=" * 50)
print("  04_anomali_temizleme TAMAMLANDI")
print("=" * 50)
print(f"  Satır : {df.shape[0]:,}")
print(f"  Sütun : {df.shape[1]}")
print()
print("  Kaydedilen artifacts:")
print("  📄 artifacts/clip_bounds.json")
print("  📄 artifacts/fill_values.json")
print("→ Sıradaki: 05_coklu_dogrusal_baginti.ipynb")

💾 Parquet kaydediliyor: ../../data/processed/lending_club_clean.parquet
✅ Kaydedildi!
   Satır  : 1,345,350
   Sütun  : 77
   Boyut  : 91.8 MB
   Süre   : 4.0 saniye
  04_anomali_temizleme TAMAMLANDI
  Satır : 1,345,350
  Sütun : 77

  Kaydedilen artifacts:
  📄 artifacts/clip_bounds.json
  📄 artifacts/fill_values.json
→ Sıradaki: 05_coklu_dogrusal_baginti.ipynb
